# Advanced Drone Technology
## Introduction to Guidance and Control Using Python Simulations

This notebook contains working simulations for:
- **Task 1**: PID Altitude Control of a Drone (with wind disturbance)
- **Task 2**: Guidance and Path Tracking of an Autonomous Boat (with water current)
- **Bonus**: Drone Figure-Eight Trajectory Tracking

Run the cells in order. Use the sliders to tune gains, click **Run Simulation and Plot**, then run the animation cell below each task.

## Setup — Run this cell first

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import ipywidgets as widgets
from IPython.display import display, HTML

plt.rcParams['figure.dpi'] = 100

---
## Task 1: PID Controller Design — Drone Altitude Control

Adjust the **P**, **I**, **D** and **Wind** sliders, then click **Run Simulation and Plot**.
- No wind is applied from 0–6 s.
- After 6 s, a constant wind force (from the slider) pushes the drone down.

Tuning order: first P, then D, then I.

In [ ]:
# ---- Task 1: simulation model ----
dt = 0.02
T_total = 15.0
time_array = np.arange(0, T_total, dt)
mass = 1.0              # kg
g = 9.81                # m/s^2
target_altitude = 10.0  # m, desired hold altitude
wind_start_time = 6.0   # s, wind is introduced after this time

# globals used by the animation cell below (populated after you click Run)
sim_time = None
sim_altitude = None
sim_target = target_altitude
sim_gains = {'P': 0, 'I': 0, 'D': 0, 'Wind': 0}

def simulate_drone_altitude(Kp, Ki, Kd, wind_force):
    """Simulate a 1-DOF drone altitude PID loop with a step wind disturbance."""
    altitude = np.zeros_like(time_array)
    velocity = np.zeros_like(time_array)
    integral_error = 0.0
    prev_error = target_altitude - altitude[0]

    for i in range(1, len(time_array)):
        error = target_altitude - altitude[i-1]
        integral_error += error * dt
        derivative_error = (error - prev_error) / dt
        prev_error = error

        # PID force, with gravity feed-forward so P/I/D only fight the error
        control_force = Kp*error + Ki*integral_error + Kd*derivative_error + mass*g

        disturbance = wind_force if time_array[i] >= wind_start_time else 0.0

        net_force = control_force - mass*g - disturbance
        acceleration = net_force / mass

        velocity[i] = velocity[i-1] + acceleration*dt
        altitude[i] = altitude[i-1] + velocity[i]*dt

        if altitude[i] < 0:      # drone can't fly below ground
            altitude[i] = 0.0
            velocity[i] = 0.0

    return altitude

print("Task 1 model ready.")

In [ ]:
# ---- Task 1: sliders + Run button ----
p_slider = widgets.FloatSlider(value=2.0, min=0.0, max=20.0, step=0.1, description='P Gain:')
i_slider = widgets.FloatSlider(value=0.0, min=0.0, max=5.0,  step=0.05, description='I Gain:')
d_slider = widgets.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.1, description='D Gain:')
wind_slider = widgets.FloatSlider(value=5.0, min=0.0, max=30.0, step=0.5, description='Wind (N):')
run_button_1 = widgets.Button(description='Run Simulation and Plot', button_style='success')
output_1 = widgets.Output()

def on_run_task1_clicked(b):
    global sim_time, sim_altitude, sim_target, sim_gains
    with output_1:
        output_1.clear_output(wait=True)
        Kp, Ki, Kd, wind_force = p_slider.value, i_slider.value, d_slider.value, wind_slider.value
        altitude = simulate_drone_altitude(Kp, Ki, Kd, wind_force)

        sim_time, sim_altitude, sim_target = time_array, altitude, target_altitude
        sim_gains = {'P': Kp, 'I': Ki, 'D': Kd, 'Wind': wind_force}

        plt.figure(figsize=(9, 5))
        plt.plot(time_array, altitude, label='Drone Altitude', color='tab:blue', linewidth=2)
        plt.axhline(target_altitude, color='green', linestyle='--', label='Target Altitude')
        plt.axvline(wind_start_time, color='red', linestyle=':', label='Wind Introduced')
        plt.title(f'PID Altitude Response  (P={Kp}, I={Ki}, D={Kd}, Wind={wind_force} N)')
        plt.xlabel('Time (s)'); plt.ylabel('Altitude (m)')
        plt.legend(); plt.grid(True)
        plt.show()

        overshoot = max(0, np.max(altitude) - target_altitude)
        final_error = abs(altitude[-1] - target_altitude)
        print(f"Max overshoot: {overshoot:.3f} m   |   Final tracking error: {final_error:.3f} m")

run_button_1.on_click(on_run_task1_clicked)
display(widgets.VBox([p_slider, i_slider, d_slider, wind_slider, run_button_1, output_1]))

Once you're happy with the response above, run the next cell to animate the tuned drone.

In [ ]:
# ---- Task 1: animation of the tuned run ----
if sim_altitude is None:
    print("Click 'Run Simulation and Plot' in the cell above at least once first.")
else:
    fig, ax = plt.subplots(figsize=(3, 6))
    ax.set_xlim(-1, 1)
    ax.set_ylim(0, max(sim_target*1.5, np.max(sim_altitude)*1.1))
    ax.axhline(sim_target, color='green', linestyle='--', label='Target Altitude')
    ax.set_ylabel('Altitude (m)')
    ax.set_title('Drone Altitude Animation')
    ax.legend(loc='upper right')
    ax.set_xticks([])

    drone_point, = ax.plot([], [], 'o', markersize=22, color='tab:blue')
    trail_line,  = ax.plot([], [], color='tab:blue', alpha=0.3)

    frame_skip = 4
    frames = range(0, len(sim_time), frame_skip)

    def init_anim():
        drone_point.set_data([], [])
        trail_line.set_data([], [])
        return drone_point, trail_line

    def update_anim(idx):
        drone_point.set_data([0], [sim_altitude[idx]])
        trail_line.set_data(np.zeros(idx+1), sim_altitude[:idx+1])
        return drone_point, trail_line

    ani = animation.FuncAnimation(fig, update_anim, frames=frames,
                                   init_func=init_anim, interval=30, blit=True)
    plt.close(fig)
    display(HTML(ani.to_jshtml()))

---
## Task 2: Guidance and Path Tracking — Autonomous Boat

The boat must follow the **blue dotted** desired path while water current pushes it off course.
Tune **kp**, **kd**, and the **current_x / current_y** sliders, then click **Run Simulation and Plot**.
Both panels show the boat's actual track vs the desired path — one with current, one without.

In [ ]:
# ---- Task 2: simulation model ----
dt2 = 0.05
T2_total = 40.0
n_steps2 = int(T2_total/dt2)
path_amplitude = 5.0
path_freq = 0.15
boat_speed = 2.0  # m/s, constant forward speed

def desired_path_y(x):
    return path_amplitude * np.sin(path_freq * x)

def desired_path_slope(x):
    return path_amplitude * path_freq * np.cos(path_freq * x)

def simulate_boat(kp, kd, current_x, current_y):
    """Pure-pursuit style PD guidance law with cross-track correction."""
    x = np.zeros(n_steps2)
    y = np.zeros(n_steps2)
    psi = np.zeros(n_steps2)   # heading angle
    prev_heading_error = 0.0

    for i in range(1, n_steps2):
        desired_y = desired_path_y(x[i-1])
        slope = desired_path_slope(x[i-1])
        desired_heading = np.arctan2(slope, 1.0)

        cross_track_error = desired_y - y[i-1]
        heading_error = desired_heading - psi[i-1] + 0.3*np.arctan2(cross_track_error, 2.0)
        d_heading_error = (heading_error - prev_heading_error) / dt2
        prev_heading_error = heading_error

        turn_rate = kp*heading_error + kd*d_heading_error
        psi[i] = psi[i-1] + turn_rate*dt2

        x[i] = x[i-1] + (boat_speed*np.cos(psi[i]) + current_x)*dt2
        y[i] = y[i-1] + (boat_speed*np.sin(psi[i]) + current_y)*dt2

    return x, y

print("Task 2 model ready.")

In [ ]:
# ---- Task 2: sliders + Run button ----
kp_slider = widgets.FloatSlider(value=1.5, min=0.0, max=10.0, step=0.1, description='kp:')
kd_slider = widgets.FloatSlider(value=0.5, min=0.0, max=5.0,  step=0.05, description='kd:')
current_x_slider = widgets.FloatSlider(value=0.3, min=-2.0, max=2.0, step=0.05, description='current_x:')
current_y_slider = widgets.FloatSlider(value=0.2, min=-2.0, max=2.0, step=0.05, description='current_y:')
run_button_2 = widgets.Button(description='Run Simulation and Plot', button_style='success')
output_2 = widgets.Output()

def on_run_task2_clicked(b):
    with output_2:
        output_2.clear_output(wait=True)
        kp, kd = kp_slider.value, kd_slider.value
        cx, cy = current_x_slider.value, current_y_slider.value

        bx, by = simulate_boat(kp, kd, cx, cy)      # with current
        bx0, by0 = simulate_boat(kp, kd, 0.0, 0.0)  # without current (ideal)

        max_x = max(bx.max(), bx0.max()) + 5
        path_x = np.linspace(0, max_x, 500)
        path_y = desired_path_y(path_x)

        fig, axes = plt.subplots(1, 2, figsize=(13, 5))

        axes[0].plot(path_x, path_y, 'b--', label='Desired Path')
        axes[0].plot(bx0, by0, color='tab:orange', label='Boat Path')
        axes[0].set_title('Without Disturbance / Current')
        axes[0].set_xlabel('X (m)'); axes[0].set_ylabel('Y (m)')
        axes[0].legend(); axes[0].grid(True); axes[0].axis('equal')

        axes[1].plot(path_x, path_y, 'b--', label='Desired Path')
        axes[1].plot(bx, by, color='tab:red', label='Boat Path')
        axes[1].set_title(f'With Current (current_x={cx}, current_y={cy})')
        axes[1].set_xlabel('X (m)'); axes[1].set_ylabel('Y (m)')
        axes[1].legend(); axes[1].grid(True); axes[1].axis('equal')

        plt.tight_layout()
        plt.show()

        print(f"kp={kp}, kd={kd}, current_x={cx}, current_y={cy}")
        print(f"Mean cross-track error (with current): {np.mean(np.abs(by - desired_path_y(bx))):.3f} m")

run_button_2.on_click(on_run_task2_clicked)
display(widgets.VBox([kp_slider, kd_slider, current_x_slider, current_y_slider, run_button_2, output_2]))

### Final animation
Manually type in the **kp, kd, current_x, current_y** values you found above, then run this cell to
produce the final boat guidance animation. Remember: for the *without disturbance* case, set
`current_x = 0` and `current_y = 0`.

In [ ]:
# ---- Task 2: final animation (EDIT these 4 values based on your tuning) ----
kp = 1.5
kd = 0.5
current_x = 0.3
current_y = 0.2

bx, by = simulate_boat(kp, kd, current_x, current_y)
max_x = bx.max() + 5
path_x = np.linspace(0, max_x, 500)
path_y = desired_path_y(path_x)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(path_x, path_y, 'b--', label='Desired Path')
boat_dot, = ax.plot([], [], 'o', color='tab:red', markersize=10, label='Boat')
trail, = ax.plot([], [], color='tab:red', alpha=0.4)
ax.set_xlim(path_x.min()-2, path_x.max()+2)
ax.set_ylim(min(path_y.min(), by.min())-2, max(path_y.max(), by.max())+2)
ax.set_title(f'Boat Guidance Animation (kp={kp}, kd={kd}, current=({current_x}, {current_y}))')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.legend(); ax.grid(True); ax.axis('equal')

frame_skip2 = 4
frames2 = range(0, len(bx), frame_skip2)

def init2():
    boat_dot.set_data([], [])
    trail.set_data([], [])
    return boat_dot, trail

def update2(idx):
    boat_dot.set_data([bx[idx]], [by[idx]])
    trail.set_data(bx[:idx+1], by[:idx+1])
    return boat_dot, trail

ani2 = animation.FuncAnimation(fig, update2, frames=frames2, init_func=init2, interval=30, blit=True)
plt.close(fig)
display(HTML(ani2.to_jshtml()))

---
## Creative Bonus Task: Drone Figure-Eight Trajectory

A drone tracks a figure-eight (lemniscate) reference path using a PD waypoint-tracking
controller, with a small constant wind disturbance added to both axes.

In [ ]:
# ---- Bonus: figure-eight tracking simulation ----
dt3 = 0.02
T3_total = 20.0
n_steps3 = int(T3_total/dt3)
t3 = np.arange(n_steps3)*dt3

A = 5.0        # figure-eight amplitude (m)
omega = 0.5    # angular frequency (rad/s)

def figure_eight_waypoint(t):
    x = A*np.sin(omega*t)
    y = A*np.sin(omega*t)*np.cos(omega*t)
    return x, y

# PD gains for position tracking + constant wind disturbance
Kp_pos = 4.0
Kd_pos = 2.0
wind_disturb_x = 0.3
wind_disturb_y = -0.2

drone_x = np.zeros(n_steps3); drone_y = np.zeros(n_steps3)
vel_x   = np.zeros(n_steps3); vel_y   = np.zeros(n_steps3)
des_x_arr = np.zeros(n_steps3); des_y_arr = np.zeros(n_steps3)

for i in range(1, n_steps3):
    des_x, des_y = figure_eight_waypoint(t3[i])
    des_x_arr[i], des_y_arr[i] = des_x, des_y

    error_x = des_x - drone_x[i-1]
    error_y = des_y - drone_y[i-1]

    accel_x = Kp_pos*error_x - Kd_pos*vel_x[i-1] + wind_disturb_x
    accel_y = Kp_pos*error_y - Kd_pos*vel_y[i-1] + wind_disturb_y

    vel_x[i] = vel_x[i-1] + accel_x*dt3
    vel_y[i] = vel_y[i-1] + accel_y*dt3
    drone_x[i] = drone_x[i-1] + vel_x[i]*dt3
    drone_y[i] = drone_y[i-1] + vel_y[i]*dt3

plt.figure(figsize=(7, 7))
plt.plot(des_x_arr, des_y_arr, 'b--', label='Desired Figure-Eight Path')
plt.plot(drone_x, drone_y, color='tab:purple', label='Drone Tracked Path')
plt.title('Bonus: Drone Figure-Eight Trajectory Tracking')
plt.xlabel('X (m)'); plt.ylabel('Y (m)')
plt.legend(); plt.grid(True); plt.axis('equal')
plt.show()

In [ ]:
# ---- Bonus: figure-eight animation ----
fig3, ax3 = plt.subplots(figsize=(7, 7))
ax3.plot(des_x_arr, des_y_arr, 'b--', label='Desired Path')
drone_dot, = ax3.plot([], [], 'o', color='tab:purple', markersize=10, label='Drone')
trail3, = ax3.plot([], [], color='tab:purple', alpha=0.4)
ax3.set_xlim(des_x_arr.min()-2, des_x_arr.max()+2)
ax3.set_ylim(des_y_arr.min()-2, des_y_arr.max()+2)
ax3.set_title('Drone Figure-Eight Animation')
ax3.set_xlabel('X (m)'); ax3.set_ylabel('Y (m)')
ax3.legend(); ax3.grid(True); ax3.axis('equal')

frame_skip3 = 5
frames3 = range(0, n_steps3, frame_skip3)

def init3():
    drone_dot.set_data([], [])
    trail3.set_data([], [])
    return drone_dot, trail3

def update3(idx):
    drone_dot.set_data([drone_x[idx]], [drone_y[idx]])
    trail3.set_data(drone_x[:idx+1], drone_y[:idx+1])
    return drone_dot, trail3

ani3 = animation.FuncAnimation(fig3, update3, frames=frames3, init_func=init3, interval=30, blit=True)
plt.close(fig3)
display(HTML(ani3.to_jshtml()))